In [8]:
import cv2
import numpy as np
import time
import winsound
import threading
from ultralytics import YOLO
import mediapipe as mp

# ==========================================
# 1. CÀI ĐẶT LUỒNG ÂM THANH (KHÔNG LÀM ĐƠ CAMERA)
# ==========================================
alarm_active = False

def play_alarm():
    global alarm_active
    while True:
        if alarm_active:
            # Phát âm bíp 500ms, tần số 1200Hz
            winsound.Beep(1200, 500)
            time.sleep(0.1) # Nghỉ 100ms
        else:
            time.sleep(0.1) # Ngủ chờ nếu không có báo động

# Khởi chạy luồng âm thanh chạy ngầm
alarm_thread = threading.Thread(target=play_alarm, daemon=True)
alarm_thread.start()

# ==========================================
# 2. KHỞI TẠO MODEL VÀ BIẾN
# ==========================================
model = YOLO("yolov8n.pt")

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

# Dictionary lưu trạng thái của từng người theo Tracking ID
track_history = {}

def get_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])

In [9]:
# ==========================================
# 3. VÒNG LẶP XỬ LÝ CHÍNH
# ==========================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_time = time.time()
    any_fall_detected = False # Cờ kiểm tra xem có ai đang ngã không

    # Dùng model.track để theo dõi ID từng người
    results = model.track(frame, persist=True, classes=[0], verbose=False)

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            bbox = (x1, y1, x2, y2)

            # Khởi tạo trạng thái cho người mới
            if track_id not in track_history:
                track_history[track_id] = {
                    'prev_center': None,
                    'prev_time': current_time,
                    'high_drop_speed_detected': False, # Cờ ghi nhận có rơi nhanh không
                    'fall_confirmed': False
                }

            state = track_history[track_id]
            center = get_center(bbox)

            # ------------------------------------------------
            # BƯỚC A: TÍNH VẬN TỐC RƠI (TRỤC Y) ĐỂ PHÂN BIỆT NGÃ & NẰM CHỦ ĐỘNG
            # ------------------------------------------------
            dt = current_time - state['prev_time']
            vertical_velocity = 0 

            if state['prev_center'] is not None and dt > 0:
                dy_center = center[1] - state['prev_center'][1] # Chỉ lấy độ lệch trục Y
                vertical_velocity = dy_center / dt # Vận tốc dọc (Pixel / giây)

            state['prev_center'] = center
            state['prev_time'] = current_time

            # Nếu tốc độ rớt xuống trục Y vượt ngưỡng -> Đánh dấu là có cú rơi đột ngột
            # Ngưỡng 300 có thể tinh chỉnh to/nhỏ tùy camera của bạn
            if vertical_velocity > 50:
                state['high_drop_speed_detected'] = True

            # ------------------------------------------------
            # BƯỚC B: MEDIA PIPE - KIỂM TRA TƯ THẾ (VAI & HÔNG)
            # ------------------------------------------------
            person_crop = frame[y1:y2, x1:x2]
            is_fallen_posture = False

            if person_crop.size != 0:
                rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
                pose_result = pose.process(rgb)

                if pose_result.pose_landmarks:
                    mp_draw.draw_landmarks(person_crop, pose_result.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                    
                    lm = pose_result.pose_landmarks.landmark
                    # Lấy Y trung bình của 2 vai và 2 hông (từ 0.0 đến 1.0)
                    shoulder_y = (lm[11].y + lm[12].y) / 2 
                    hip_y = (lm[23].y + lm[24].y) / 2      

                    # Nếu Y_vai > Y_hông (tức là vai đang gần mặt đất hơn hông)
                    # Trừ hao 0.1 để bắt cả trường hợp đầu hơi ngóc lên
                    if shoulder_y > (hip_y - 0.2):
                        is_fallen_posture = True

            frame[y1:y2, x1:x2] = person_crop

            # ------------------------------------------------
            # BƯỚC C: KẾT HỢP LOGIC & CẬP NHẬT GIAO DIỆN
            # ------------------------------------------------
            color = (0, 255, 0) # Xanh lá: Bình thường

            # NGÃ = Tư thế nằm (Vai thấp hơn hông) + Trước đó có rơi tự do
            if is_fallen_posture and state['high_drop_speed_detected']:
                color = (0, 0, 255) # Đỏ
                state['fall_confirmed'] = True
                any_fall_detected = True
            
            # ĐỨNG DẬY = Không còn tư thế nằm (Vai cao hơn hông) -> Xóa mọi cảnh báo
            elif not is_fallen_posture:
                color = (0, 255, 0)
                state['fall_confirmed'] = False
                state['high_drop_speed_detected'] = False # Xóa cờ rơi

            # Vẽ khung và thông số
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            info_text = f"ID:{track_id} VelY:{vertical_velocity:.0f} Fall:{state['fall_confirmed']}"
            cv2.putText(frame, info_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Bật/tắt chuông báo dựa trên việc có người nào đang bị ngã không
    global alarm_active
    alarm_active = any_fall_detected

    cv2.imshow("Smart Fall Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Dọn dẹp khi thoát
alarm_active = False 
cap.release()
cv2.destroyAllWindows()